In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
from rdkit import Chem
from rdkit.Geometry import Point3D
import mdtraj as md
from pathlib import Path
from rdkit.Chem import rdFMCS
from rdkit.Chem import rdchem
from rdkit.Chem import rdmolops
from rdkit.Chem import CanonicalRankAtoms

In [20]:
# Set paths
mol_dir = Path("../data/dynamic_900")
converted_dir = Path("../data/dynamic_900/converted")
converted_dir.mkdir(exist_ok=True)

# Load reference molecule
ref_mol_path = mol_dir / "molecule_000.mol"
ref_mol = Chem.MolFromMolFile(str(ref_mol_path), removeHs=False, sanitize=False)
ref_symbols = [atom.GetSymbol() for atom in ref_mol.GetAtoms()]

def reorder_atoms_to_match_reference(mol, ref_mol):
    ref_ranks = CanonicalRankAtoms(ref_mol)
    mol_ranks = CanonicalRankAtoms(mol)

    # Map mol atoms to ref atoms based on canonical ranking
    rank_to_index = {rank: i for i, rank in enumerate(mol_ranks)}
    reorder = [rank_to_index[r] for r in ref_ranks]

    reordered = Chem.RenumberAtoms(mol, reorder)
    return reordered

def extract_coordinates(mol):
    conf = mol.GetConformer()
    return [conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())]

for mol_path in sorted(mol_dir.glob("molecule_*.mol")):
    mol = Chem.MolFromMolFile(str(mol_path), removeHs=False, sanitize=False)
    
    try:
        reordered = reorder_atoms_to_match_reference(mol, ref_mol)
        reordered_symbols = [atom.GetSymbol() for atom in reordered.GetAtoms()]
        assert reordered_symbols == ref_symbols, f"Mismatch in atom types: {mol_path.name}"
    except Exception as e:
        print(f"Skipping {mol_path.name}: {e}")
        continue

    coords = extract_coordinates(reordered)
    new_mol = Chem.Mol(ref_mol)
    
    new_conf = Chem.Conformer(new_mol.GetNumAtoms())
    for i, pos in enumerate(coords):
        new_conf.SetAtomPosition(i, Point3D(pos.x, pos.y, pos.z))
    new_mol.RemoveAllConformers()
    new_mol.AddConformer(new_conf)

    out_path = converted_dir / mol_path.with_suffix(".pdb").name
    Chem.MolToPDBFile(new_mol, str(out_path))
    print(f"Written: {out_path.name}")

# Optional: combine into trajectory
pdb_files = sorted(converted_dir.glob("*.pdb"))
traj = md.load([str(p) for p in pdb_files])
traj.save(converted_dir / "trajectory.dcd")
traj[0].save(converted_dir / "topology.pdb")
print("Saved trajectory.dcd and topology.pdb")


Written: molecule_000.pdb
Written: molecule_001.pdb
Written: molecule_002.pdb
Written: molecule_003.pdb
Written: molecule_004.pdb
Written: molecule_005.pdb
Written: molecule_006.pdb
Written: molecule_007.pdb
Written: molecule_008.pdb
Written: molecule_009.pdb
Written: molecule_010.pdb
Written: molecule_011.pdb
Written: molecule_012.pdb
Written: molecule_013.pdb
Written: molecule_014.pdb
Written: molecule_015.pdb
Written: molecule_016.pdb
Written: molecule_017.pdb
Written: molecule_018.pdb
Written: molecule_019.pdb
Written: molecule_020.pdb
Written: molecule_021.pdb
Written: molecule_022.pdb
Written: molecule_023.pdb
Written: molecule_024.pdb
Written: molecule_025.pdb
Written: molecule_026.pdb
Written: molecule_027.pdb
Written: molecule_028.pdb
Written: molecule_029.pdb
Written: molecule_030.pdb
Written: molecule_031.pdb
Written: molecule_032.pdb
Written: molecule_033.pdb
Written: molecule_034.pdb
Written: molecule_035.pdb
Written: molecule_036.pdb
Written: molecule_037.pdb
Written: mol

In [5]:
mol_dir = Path("../data/dynamic_900")
pdb_dir = Path("../data/dynamic_900/converted")
pdb_dir.mkdir(parents=True, exist_ok=True)

# Use this .mol file as the topology template
ref_mol_path = Path("../data/dynamic_900/molecule_000.mol")
ref_mol = Chem.MolFromMolFile(str(ref_mol_path), removeHs=False, sanitize=False)
conf = ref_mol.GetConformer()

def extract_coordinates(mol):
    conf = mol.GetConformer()
    return [conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())]

for mol_path in sorted(mol_dir.glob("*.mol")):
    mol = Chem.MolFromMolFile(str(mol_path), removeHs=False, sanitize=False)
    if mol is None:
        raise ValueError("No molecule")

    coords = extract_coordinates(mol)
    if len(coords) != ref_mol.GetNumAtoms():
        print(f"Atom count mismatch in {mol_path.name}")
        continue

    # Create a deep copy of the reference mol
    new_mol = Chem.Mol(ref_mol)
    
    # Create a new conformer with fresh coordinates
    new_conf = Chem.Conformer(new_mol.GetNumAtoms())
    for i, pos in enumerate(coords):
        new_conf.SetAtomPosition(i, Point3D(pos.x, pos.y, pos.z))

    new_mol.RemoveAllConformers()
    new_mol.AddConformer(new_conf, assignId=True)

    out_path = pdb_dir / mol_path.with_suffix(".pdb").name
    Chem.MolToPDBFile(new_mol, str(out_path))

print("Written all files")



Written all files


In [6]:
pdb_files = sorted(pdb_dir.glob("*.pdb"))

# Load all PDB files as frames of a trajectory
traj = md.load([str(p) for p in pdb_files])

# Save to a trajectory format (e.g., .xtc or .dcd or even .pdb for multi-frame)
traj.save(pdb_dir / "ensemble.pdb")
traj[0].save(pdb_dir / "topology.pdb")
traj.save(pdb_dir / "ensemble.dcd")